# Projet : Prédiction du Risque d'Incendies de Forêt en France

## Phase 1 : Ingestion, Nettoyage et Consolidation Spatiale

---

### Introduction et Contexte
Ce projet s'inscrit dans une démarche de modélisation prédictive du risque d'incendies de forêt sur le territoire français en couplant des données historiques de sinistres avec des variables météorologiques. L'objectif final (Phase 2) est de construire un modèle de Machine Learning capable d'évaluer la probabilité de départ de feu à l'échelle communale.

### Objectifs 
1. **Collecter et assembler** l'historique complet des incendies (1973–2026) issus de la base BDIFF.
2. **Gérer les contraintes techniques d'exportation** (limitation à 30 000 lignes par extraction) via une stratégie de fusion de blocs.
3. **Apparier ces données avec le référentiel géographique** officiel des communes françaises (INSEE 2026) pour récupérer les coordonnées spatiales (Latitude/Longitude) indispensables à l'intégration météo ultérieure.
4. **Sauvegarder** le jeu de données consolidé dans un format industriel standard (Parquet).

## 1. Chargement et Assemblage des Données Brutes BDIFF (1973-2026)

L'extraction depuis la plateforme BDIFF a nécessité un découpage temporel en 5 blocs distincts pour contourner la limite de téléchargement. Nous allons scanner le dossier `data/raw/` pour lister ces fichiers et préparer leur fusion.

In [ ]:
import glob
import os
import pandas as pd

# chemin 
RAW_DATA_DIR = os.path.join("..", "data", "raw")

# Fichiers 'Incendies_'
fichiers_incendies = glob.glob(os.path.join(RAW_DATA_DIR, "Incendies_*.csv"))
print(f"Fichiers trouvés : {fichiers_incendies}")

Fichiers trouvés : ['..\\data\\raw\\Incendies_1883.1992.csv', '..\\data\\raw\\Incendies_1973.1982.csv', '..\\data\\raw\\Incendies_1993.2003.csv', '..\\data\\raw\\Incendies_2004.2015.csv', '..\\data\\raw\\Incendies_2016.2026.csv']


## 2. Ingestion itérative et harmonisation des fichiers

Chaque export BDIFF possède des en-têtes et des commentaires métadonnées spécifiques en début de fichier (lignes parasites). Nous appliquons une logique de lecture adaptative (`skiprows`) selon les périodes, forçons le typage textuel du Code INSEE pour éviter la perte des zéros initiaux (ex: `01001`), et concaténons le tout.

In [ ]:
liste_df = []

for f in fichiers_incendies:
    # Ajuste le nombre de lignes à sauter selon le fichier
    if "2004.2015" in f:
        skip = 6  
    elif "1973.1982" in f:
        skip = 2
    else:
        skip = 3

    print(f"Chargement de {f} (skip={skip})...")

    # Lecture du fichier et forçage du type pour le Code INSEE
    df_temp = pd.read_csv(
        f, skiprows=skip, sep=";", dtype={"Code INSEE": str, "Département": str}
    )
    liste_df.append(df_temp)

# Fusionner tous les blocs en un seul DataFrame unique
df_incendies = pd.concat(liste_df, ignore_index=True)

# Nettoyage cosmétique : On force le code INSEE à avoir 5 caractères (ex: '1001' devient '01001')
df_incendies["Code INSEE"] = (
    df_incendies["Code INSEE"].astype(str).str.zfill(5)
)

print("-" * 40)
print(f"🔥 Succès ! Nombre total d'incendies cumulés : {len(df_incendies)}")

Chargement de ..\data\raw\Incendies_1883.1992.csv (skip=3)...
Chargement de ..\data\raw\Incendies_1973.1982.csv (skip=2)...
Chargement de ..\data\raw\Incendies_1993.2003.csv (skip=3)...
Chargement de ..\data\raw\Incendies_2004.2015.csv (skip=6)...
Chargement de ..\data\raw\Incendies_2016.2026.csv (skip=3)...
----------------------------------------
🔥 Succès ! Nombre total d'incendies cumulés : 134117


* **Volume collecté :** Le script a assemblé avec succès les 5 blocs pour un total de **134 117 lignes d'incendies**.
* **Intégrité :** La structure globale est conservée, et la colonne `Code INSEE` a été normalisée sur 5 caractères (`str.zfill(5)`), ce qui garantit la viabilité de la future jointure.

## 3. Enrichissement Spatial : Couplage avec le Référentiel des Communes

Pour géolocaliser nos incendies, nous chargeons le fichier unique `communes-france-2026.csv`. Nous sélectionnons uniquement les variables administratives et géographiques indispensables (Population, Densité, Coordonnées du centre de la commune, Altitude moyenne). Une jointure gauche (`left join`) est réalisée en utilisant le `Code INSEE` comme clé de liaison.

In [ ]:
# 1. Chemin 
path_communes = os.path.join(RAW_DATA_DIR, "communes-france-2026.csv")

# 2. Chargement en forçant le code INSEE en texte pour ne pas perdre les zéros initiaux
df_communes = pd.read_csv(path_communes, dtype={"code_insee": str})

# Uniformisation du code INSEE des communes
df_communes["code_insee"] = df_communes["code_insee"].astype(str).str.zfill(5)

# 3. Sélection des colonnes stratégiques pour notre étude spatio-temporelle et notre modèle
colonnes_utiles = [
    "code_insee",
    "nom_standard",
    "reg_nom",
    "dep_nom",
    "population",
    "superficie_km2",
    "densite",
    "altitude_moyenne",
    "latitude_centre",
    "longitude_centre",
]

# On restreint le dataframe aux colonnes sélectionnées
df_communes_clean = df_communes[colonnes_utiles]

print(f"📍 Référentiel chargé : {len(df_communes_clean)} communes prêtes.")

# 4. JOINTURE 
df_consolide = pd.merge(
    df_incendies,
    df_communes_clean,
    left_on="Code INSEE",
    right_on="code_insee",
    how="left",
)

print("-" * 40)
print(
    f"✅ Fusion réussie ! Le dataset consolidé contient {len(df_consolide)} lignes et {len(df_consolide.columns)} colonnes."
)

C:\Users\user\AppData\Local\Temp\ipykernel_3192\1713862770.py:8: DtypeWarning: Columns (0: dep_code, 1: canton_code, 2: epci_code, 3: code_insee_centre_zone_emploi, 4: code_unite_urbaine, 5: code_insee_centre_aire_attraction, 6: code_bassin_de_vie) have mixed types. Specify dtype option on import or set low_memory=False.
  df_communes = pd.read_csv(path_communes, dtype={"code_insee": str})


📍 Référentiel chargé : 34868 communes prêtes.
----------------------------------------
✅ Fusion réussie ! Le dataset consolidé contient 134117 lignes et 34 colonnes.


## 4. Contrôle Qualité de la Consolidation

Avant de finaliser la Phase 1, nous mesurons le taux d'appariement. Les lignes d'incendies n'ayant pas trouvé de correspondance géographique représentent des anomalies (communes supprimées, fusionnées ou anciennes). Un taux de perte marginal est toléré.

In [ ]:
# Le nombre d'incendies qui n'ont pas trouvé de correspondance géographique
manquants = df_consolide["latitude_centre"].isna().sum()
pourcentage_manquants = (manquants / len(df_consolide)) * 100

print(f"⚠️ Nombre d'incendies sans coordonnées GPS : {manquants}")
print(f"📊 Soit un taux de perte de : {pourcentage_manquants:.2f}%")

df_consolide[
    ["Année", "Nom de la commune", "latitude_centre", "longitude_centre"]
].head()

⚠️ Nombre d'incendies sans coordonnées GPS : 874
📊 Soit un taux de perte de : 0.65%


,Année,Nom de la commune,latitude_centre,longitude_centre
0,1983,Saint-Vallier-de-Thiey,43.697,6.851
1,1983,Mons,43.699,6.709
2,1983,Bagnols-en-Forêt,43.533,6.708
3,1983,La Mure-Argens,44.017,6.532
4,1983,Rabouillet,42.727,2.371


### 📊 Diagnostic de la Jointure
* **Taux de perte :** Seulement **0,65%** des incendies (874 lignes sur 134 117) n'ont pas de coordonnées GPS.

Ce taux est extrêmement faible et témoigne de la haute cohérence entre la base BDIFF historique et le référentiel 2026. Le dataset est considéré comme géographiquement fiable. Les lignes sans coordonnées pourront être écartées ou imputées à l'échelle départementale lors de la phase de Feature Engineering.

## 5. Persistance du Dataset au Format Industriel

Le fichier final consolidé (données incendies + données géographiques) est exporté dans le sous-dossier `data/processed/` au format **Parquet**. Ce format colonnaire et compressé préserve strictement les types de données (notamment les chaînes de caractères des codes INSEE) et optimise les performances de lecture pour la Phase 2.

In [ ]:
# Crée le dossier 'processed' 
PROCESSED_DATA_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

# Chemin de sauvegarde
output_path = os.path.join(PROCESSED_DATA_DIR, "bdiff_consolidee_phase1.parquet")

# Sauvegarde au format industriel Parquet
df_consolide.to_parquet(output_path, index=False)

print(f"💾 Dataset de la Phase 1 sauvegardé avec succès dans : {output_path}")

💾 Dataset de la Phase 1 sauvegardé avec succès dans : ..\data\processed\bdiff_consolidee_phase1.parquet


In [ ]:
# On indique le chemin exact vers le fichier
chemin_fichier = r"C:\Users\user\Documents\projets\projet_plateforme\projet_plateforme\Projet_Terre-vent-feu-eau-data\data\processed\bdiff_consolidee_phase1.parquet"

# Lecture du  fichier
df = pd.read_parquet(chemin_fichier)

print(df.head())

   Année  Numéro Département Code INSEE       Nom de la commune  \
0   1983      89          06      06130  Saint-Vallier-de-Thiey   
1   1983    2210          83      83080                    Mons   
2   1983    2211          83      83008        Bagnols-en-Forêt   
3   1983       1          04      04136          La Mure-Argens   
4   1983    2018          66      66156              Rabouillet   

  Date de première alerte  Surface parcourue (m2)  Surface forêt (m2)  \
0     1983-01-01 19:00:00                 18000.0              8460.0   
1     1983-01-02 12:40:00                  2000.0               940.0   
2     1983-01-03 12:00:00                 10000.0              4700.0   
3     1983-01-03 12:10:00                 15000.0              7050.0   
4     1983-01-03 13:00:00                100000.0             47000.0   

   Surface maquis garrigues (m2)  Autres surfaces naturelles hors forêt (m2)  \
0                         8640.0                                         0.0  